# YouTube Comment Extractor Code
The underlying code will get a text file as an input which will contain list of YouTube urls. Based on the URL, the code will detect whether it's a playlist, a channel or a video URL. Based on this, further procesing will happen and comments for every video will be saved in a csv file in format "comments_videoID.csv". On a colab CPU environment, it took around 12s to extract the comments for 10 videos. Hence it can be inferred that the compute requirements are not so high for this code.

**Caution**: YouTube Data API v3 is free to use however they have a daily quota of 10000 units. Based on certain request, they get consumed. To know more, [click](https://developers.google.com/youtube/v3/determine_quota_cost).

For a wider usage, it is advisable to create and use multiple API keys in shuffling.

I hope this helps :)

In [1]:
import csv
import re
from googleapiclient.discovery import build
import html
import os
from pathlib import Path

# Load API key from .env file
def load_api_key():
    """Load YouTube API key from .env file."""
    env_path = Path(__file__).parent / '.env' if '__file__' in globals() else Path('.env')
    
    if env_path.exists():
        with open(env_path, 'r') as f:
            for line in f:
                if line.startswith('YOUTUBE_API_KEY='):
                    return line.split('=', 1)[1].strip()
    
    # Fallback to environment variable
    api_key = os.getenv('YOUTUBE_API_KEY')
    if not api_key:
        raise ValueError("API key not found. Please set YOUTUBE_API_KEY in .env file or environment variable.")
    return api_key

API_KEY = load_api_key()

# FULL MODE: Collect up to 20,000 Hinglish comments
COMMENT_LIMIT = 20000

# Common Hinglish words in Roman script (WhatsApp style)
HINGLISH_WORDS = {
    'hai', 'ho', 'ka', 'ki', 'ko', 'ke', 'se', 'me', 'mein', 'par', 'ke', 'kya', 
    'yaar', 'bhai', 'dost', 'sab', 'kuch', 'aur', 'bhi', 'tha', 'thi', 'the', 
    'hain', 'tha', 'matlab', 'achha', 'accha', 'theek', 'thik', 'nahi', 'nahin',
    'kaise', 'kaisa', 'kyu', 'kyun', 'kyuki', 'kyunki', 'lekin', 'par', 'acha',
    'wala', 'wali', 'wale', 'ji', 'sir', 'madam', 'sahab', 'bol', 'bolna', 'bola',
    'dekh', 'dekho', 'dekha', 'suno', 'suna', 'karo', 'karna', 'gaya', 'gayi', 'gaye',
    'liya', 'liye', 'diya', 'diye', 'hua', 'hui', 'hue', 'raha', 'rahe', 'rahi',
    'chaliye', 'chalo', 'aao', 'jao', 'karo', 'bahut', 'bohot', 'bohat', 'sabhi',
    'zaroor', 'jarur', 'bilkul', 'ekdum', 'waah', 'wah', 'sahi', 'galat', 'thoda',
    'bahut', 'kam', 'jyada', 'zyada', 'itna', 'utna', 'kitna', 'abhi', 'ab', 'tab',
    'phir', 'fir', 'kabhi', 'kab', 'jab', 'agar', 'toh', 'to', 'naa', 'na', 'haan',
    'haa', 'han', 'ji', 'are', 'arre', 'oye', 'bc', 'yeh', 'ye', 'wo', 'woh', 'iska',
    'uska', 'tumhara', 'mera', 'humara', 'apna', 'khud', 'sabse', 'sab',
    'acha', 'bura', 'achchha', 'burra', 'mast', 'badhiya', 'badiya', 'zabardast',
    'kamaal', 'kamal', 'gazab', 'gajab', 'dhamaal', 'dhamal', 'bhaari', 'bhari',
    'dil', 'dost', 'yaar', 'bro', 'bhai', 'sister', 'behen', 'beta', 'beti', 'bhaiya',
    'kya', 'kaise', 'kahan', 'kahaan', 'kidar', 'kidhar', 'yahan', 'yahaan', 'wahan',
    'wahaan', 'idhar', 'udhar', 'kis', 'kon', 'kaun', 'kisko', 'kisse', 'kaunsa',
    'matlab', 'yaani', 'yani', 'mtlb', 'aisa', 'waisa', 'kaisa', 'thoda', 'thodi',
}

def clean_html_text(text):
    """Remove HTML tags, URLs, emojis, and decode HTML entities from comment text."""
    if not text:
        return ""
    
    # Decode HTML entities first
    text = html.unescape(text)
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Remove URLs
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'www\.(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    
    # Remove ALL emojis and special symbols (comprehensive)
    # Remove emoji ranges
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        u"\U0001FA00-\U0001FA6F"  # Chess Symbols
        u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        u"\U00002600-\U000026FF"  # Miscellaneous Symbols
        u"\U00002700-\U000027BF"  # Dingbats
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub('', text)
    
    # Remove special symbols and arrows
    text = re.sub(r'[→←↑↓↔️⇒⇐⇑⇓↖↗↘↙▶◀▲▼■□●○★☆♥♦♣♠✓✗✔✘]', '', text)
    
    # Remove multiple spaces and trim
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def extract_video_id(url):
    """Extracts a YouTube video ID from a URL."""
    match = re.search(r"(?:v=|\/embed\/|\/\d+\/|\/vi?\/|watch\?v=|youtu\.be\/|\/v\/|\/shorts\/)([a-zA-Z0-9_-]{11})", url)
    return match.group(1) if match else None

def extract_playlist_id(url):
    """Extracts a YouTube playlist ID from a URL."""
    match = re.search(r"list=([a-zA-Z0-9_-]+)", url)
    return match.group(1) if match else None

def extract_channel_id(url, api_key):
    """Extracts a channel ID from a URL, supporting both @ChannelName and /channel/ID formats."""
    if "/channel/" in url:
        match = re.search(r"channel/([a-zA-Z0-9_-]+)", url)
        return match.group(1) if match else None

    elif "/@" in url:
        channel_name = url.split("/@")[-1].split("?")[0]
        youtube = build("youtube", "v3", developerKey=api_key)
        request = youtube.search().list(
            part="snippet",
            type="channel",
            q=channel_name,
            maxResults=1
        )
        response = request.execute()

        if "items" in response and response["items"]:
            return response["items"][0]["id"]["channelId"]

    return None

def get_video_ids_from_playlist(playlist_id, api_key):
    """Fetches all video IDs from a given YouTube playlist."""
    youtube = build("youtube", "v3", developerKey=api_key)
    video_ids = []
    next_page_token = None

    while True:
        request = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            maxResults=50,  # Max allowed by API
            pageToken=next_page_token
        )
        response = request.execute()

        for item in response["items"]:
            video_ids.append(item["contentDetails"]["videoId"])

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    return video_ids

def get_uploads_playlist_from_channel(channel_id, api_key):
    """Fetches the 'Uploads' playlist ID from a channel."""
    youtube = build("youtube", "v3", developerKey=api_key)
    request = youtube.channels().list(
        part="contentDetails",
        id=channel_id
    )
    response = request.execute()

    if "items" in response and response["items"]:
        return response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
    return None

def is_hinglish(text):
    """
    Detects if text contains code-mixed Hindi-English (Hinglish).
    Supports both Devanagari script and Roman script Hinglish.
    Ensures complete sentences (minimum length).
    """
    # Clean text first
    cleaned = clean_html_text(text)
    
    # Must be at least 15 characters for a complete sentence
    if len(cleaned) < 15:
        return False
    
    # Check for Devanagari script (Hindi)
    has_hindi = bool(re.search(r'[\u0900-\u097F]', cleaned))
    
    # Check for English letters
    has_english = bool(re.search(r'[a-zA-Z]', cleaned))
    
    # Method 1: Devanagari + English = Hinglish (complete sentence)
    if has_hindi and has_english:
        return True
    
    # Method 2: Roman script Hinglish (WhatsApp style)
    # Check for common Hinglish words in Roman script
    if has_english and not has_hindi:
        words = re.findall(r'\b[a-zA-Z]+\b', cleaned.lower())
        if len(words) < 3:  # Must have at least 3 words for a sentence
            return False
        hinglish_count = sum(1 for word in words if word in HINGLISH_WORDS)
        # If 15% or more words are common Hinglish words, consider it Hinglish
        if (hinglish_count / len(words)) >= 0.15:
            return True
    
    return False

def get_youtube_comments(video_id, api_key, max_results=100):
    """Fetches comments for a given YouTube video."""
    youtube = build("youtube", "v3", developerKey=api_key)
    comments = []
    next_page_token = None

    while True:
        try:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=max_results,
                pageToken=next_page_token
            )
            response = request.execute()

            for item in response.get("items", []):
                comment = item["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
                # Filter for Hinglish comments only
                if is_hinglish(comment):
                    # Clean the comment before saving
                    cleaned_comment = clean_html_text(comment)
                    if cleaned_comment and len(cleaned_comment) >= 15:  # Ensure complete sentence
                        comments.append(cleaned_comment)

            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break
        except Exception as e:
            print(f"  Error fetching comments: {e}")
            break

    return comments

def save_comments_to_csv(video_id, comments):
    """Saves comments to a CSV file."""
    filename = f"comments_{video_id}.csv"

    with open(filename, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["Comment"])  # Header row
        for comment in comments:
            writer.writerow([comment])

    print(f"  Saved {len(comments)} Hinglish comments to {filename}")

def process_link(url, all_comments):
    """Processes a single link (video, playlist, or channel) and scrapes comments."""
    if "playlist" in url:
        playlist_id = extract_playlist_id(url)
        if not playlist_id:
            print("  Invalid playlist URL:", url)
            return

        video_ids = get_video_ids_from_playlist(playlist_id, API_KEY)
        print(f"  Found {len(video_ids)} videos in the playlist.")

    elif "/channel/" in url or "/@" in url:
        channel_id = extract_channel_id(url, API_KEY)
        if not channel_id:
            print("  Invalid channel URL:", url)
            return

        uploads_playlist_id = get_uploads_playlist_from_channel(channel_id, API_KEY)
        if not uploads_playlist_id:
            print("  Could not find the uploads playlist for the channel:", url)
            return

        video_ids = get_video_ids_from_playlist(uploads_playlist_id, API_KEY)
        print(f"  Found {len(video_ids)} videos in the channel's uploads.")

    else:
        video_id = extract_video_id(url)
        if not video_id:
            print("  Invalid video URL:", url)
            return

        video_ids = [video_id]

    for i, video_id in enumerate(video_ids, 1):
        if len(all_comments) >= COMMENT_LIMIT:
            print(f"\n  Reached {COMMENT_LIMIT} Hinglish comments. Stopping.")
            break
            
        print(f"\n  Fetching comments for Video {i}/{len(video_ids)} (ID: {video_id})...")
        comments = get_youtube_comments(video_id, API_KEY)
        
        # Add to master list
        all_comments.extend(comments)
        print(f"  Total Hinglish comments collected so far: {len(all_comments)}")
        
        # Save individual video comments
        if comments:
            save_comments_to_csv(video_id, comments)

def main():
    filename = r"C:\YOUTUBESCRAPPER\urls.txt"  # Windows path
    
    # Parse URLs from file
    try:
        with open(filename, "r", encoding="utf-8") as file:
            lines = file.readlines()
            urls = []
            for line in lines:
                line = line.strip()
                # Skip empty lines, comments, and header row
                if not line or line.startswith("#") or line.startswith("Name of"):
                    continue
                # Extract URL from tab-separated format (column 2)
                parts = line.split("\t")
                if len(parts) >= 2 and parts[1].startswith("http"):
                    urls.append(parts[1])
    except FileNotFoundError:
        print(f"  File not found: {filename}")
        print("  Please create the file and add YouTube URLs.")
        return

    if not urls:
        print(f"  No valid URLs found in {filename}")
        print("  Please add YouTube URLs and run again.")
        return

    print(f"\n{'='*70}")
    print(f"  FULL MODE: Extracting up to {COMMENT_LIMIT:,} Hinglish comments")
    print(f"  Processing {len(urls)} channels/playlists/videos")
    print(f"  Features: Roman + Devanagari Hinglish | HTML/URL/Emoji removal")
    print(f"  Minimum sentence length: 15 characters")
    print(f"{'='*70}\n")
    
    all_comments = []
    
    for idx, url in enumerate(urls, 1):
        if len(all_comments) >= COMMENT_LIMIT:
            print(f"\n  Reached {COMMENT_LIMIT:,} comments. Stopping extraction.")
            break
        print(f"\n[{idx}/{len(urls)}] Processing: {url}")
        process_link(url, all_comments)
    
    # Save all Hinglish comments to master CSV
    if all_comments:
        master_filename = "hinglish_comments_all.csv"
        # Limit to COMMENT_LIMIT comments
        all_comments = all_comments[:COMMENT_LIMIT]
        
        with open(master_filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(["Comment"])
            for comment in all_comments:
                writer.writerow([comment])
        
        print(f"\n{'='*70}")
        print(f"  SUCCESS! Saved {len(all_comments):,} clean Hinglish comments")
        print(f"  Output file: {master_filename}")
        print(f"  All HTML tags, URLs, and emojis have been removed")
        print(f"  Comments are complete sentences with code-mixed Hindi-English")
        print(f"{'='*70}\n")
    else:
        print("\n  No Hinglish comments found.")

if __name__ == "__main__":
    main()



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\pranjal\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip
